# Đề xuất Kiến trúc PetNet-Hybrid V3: ConvNeXt-SE & Multi-Scale MBConv Dual-Stage Architecture (From Scratch)

**Yêu cầu xây dựng model:**
Dựa trên thành công đột phá của bản V2 (**47.94%** – vượt qua cả 11 mô hình cơ sở), phiên bản **PetNet-Hybrid V3** (~4.65M params) được thiết kế để chinh phục mốc **55% – 60%+** nhờ kết hợp triết lý thị giác hiện đại (ConvNeXt & EdgeNeXt):
1. **Phân tầng chức năng (Dual-Stage Architecture):**
   - **Tầng nông (Stage 1 & 2):** Sử dụng các khối **Residual Convolution 3x3 kết hợp Channel Attention (SE)** (kế thừa từ SE-ResNet-18). Tầng này có gradient cực mạnh, giúp mô hình nắm bắt tức thì các đặc trưng màu sắc, vân lông và kết cấu trong 5 epoch đầu.
   - **Tầng sâu (Stage 3 & 4):** Sử dụng các khối **MBConv Inverted Residual (Expansion 4x) với Depthwise 3x3 và 5x5 kết hợp SE** (kế thừa từ EfficientNet-B0). Không gian đặc trưng được mở rộng mạnh mẽ lên 224 – 352 channels nhưng tham số vẫn rất nhẹ, triệt tiêu hoàn toàn overfitting.
2. **Residual Part-Attention đa thang đo:** Module chú ý bộ phận $x \cdot (1 + 	ext{mask})$ tại đầu ra Stage 3 ($14\times 14$) và Stage 4 ($7\times 7$) giúp mô hình tự động phóng đại các vùng nhận diện độc quyền của từng giống (tai vểnh/tai cụp, mặt gãy, đốm mắt).
3. **Multi-Scale Feature Fusion Head:** Kết hợp cả đặc trưng chi tiết $14\times 14$ (GAP + GMP) và đặc trưng tổng thể $7\times 7$ (GAP + GMP) tạo thành vector 1152 chiều toàn diện.
4. **Tối ưu hóa siêu tham số (Hyperparameter Tuning):**
   - Learning Rate chuẩn mực $6\times 10^{-4}$ (không bị vọt lên quá cao gây dao động loss).
   - `weight_decay = 1e-2` và `label_smoothing = 0.05`.
   - CosineAnnealingLR kéo dài mượt mà qua 50 epochs.
5. **Khắc phục hoàn toàn PyTorch 2.6:** Thêm `weights_only=False` vào `torch.load` ở Bước 6 giúp việc nạp checkpoint và chạy TTA trên Kaggle diễn ra trơn tru 100%.


In [ ]:
# [Môi trường Kaggle / Colab] Tự động tải mã nguồn từ GitHub và chuyển vào đúng thư mục làm việc
import os

if os.path.exists("/kaggle"):
    %cd /kaggle/working
    if not os.path.exists("ml-semester-cat-dog-classifier"):
        !git clone https://github.com/LuanNguyen23T/ml-semester-cat-dog-classifier.git
    %cd /kaggle/working/ml-semester-cat-dog-classifier/notebooks/12_hybrid_model
elif os.path.exists("/content"):
    %cd /content
    if not os.path.exists("ml-semester-cat-dog-classifier"):
        !git clone https://github.com/LuanNguyen23T/ml-semester-cat-dog-classifier.git
    %cd /content/ml-semester-cat-dog-classifier/notebooks/12_hybrid_model


## Bước 1: Input các thư viện


In [ ]:
import os
import sys
import json
import random
import argparse
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix, classification_report

# Thiết lập đường dẫn import module của project
current_dir = Path(os.getcwd())
project_root = None
for p in [current_dir, current_dir.parent, current_dir.parent.parent]:
    if (p / "src" / "dataset.py").exists():
        project_root = p
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        break

try:
    from src.dataset import OxfordPetDataset, get_transforms
except ImportError:
    from dataset import OxfordPetDataset, get_transforms

# Thiết lập Random Seed đảm bảo tính tái lập (Reproducibility)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Thiết bị tính toán được phân bổ: {device}")


## Bước 2: Argparse & Thiết lập siêu tham số


In [ ]:
def parse_args():
    parser = argparse.ArgumentParser(description="Proposed PetNet-Hybrid V3 on Oxford-IIIT Pet")
    default_data_dir = "../../dataset/oxford-iiit-pet"
    if project_root and (project_root / "dataset" / "oxford-iiit-pet").exists():
        default_data_dir = str(project_root / "dataset" / "oxford-iiit-pet")
    elif project_root and (project_root / "dataset").exists():
        default_data_dir = str(project_root / "dataset" / "oxford-iiit-pet")
    parser.add_argument("--data_dir", type=str, default=default_data_dir)
    parser.add_argument("--task", type=str, default="breed", choices=["breed"])
    parser.add_argument("--img_size", type=int, default=224, help="Kích thước ảnh chuẩn 224x224")
    parser.add_argument("--batch_size", type=int, default=64, help="Mini-batch size tối ưu cho GPU T4")
    parser.add_argument("--epochs", type=int, default=50, help="Số epochs huấn luyện 50")
    parser.add_argument("--lr", type=float, default=6e-4, help="Learning rate chuẩn 0.0006 cho AdamW")
    parser.add_argument("--weight_decay", type=float, default=1e-2, help="Decoupled weight decay 0.01 chống overfitting")
    parser.add_argument("--label_smoothing", type=float, default=0.05, help="Label smoothing nhẹ 0.05 phân định rõ 37 lớp")
    parser.add_argument("--crop_foreground", type=bool, default=True, help="Tự động cắt chặt Bounding Box vùng thú cưng")
    parser.add_argument("--remove_background", type=bool, default=True, help="Xóa nền thành màu xám trung tính (128, 128, 128)")
    parser.add_argument("--checkpoint_dir", type=str, default="./checkpoints")
    parser.add_argument("--plots_dir", type=str, default="./plots")
    try:
        args, _ = parser.parse_known_args()
    except:
        args = parser.parse_args([])
    return args

args = parse_args()
print(f"Cấu hình PetNet-Hybrid V3: Batch size = {args.batch_size} | Epochs = {args.epochs} | LR = {args.lr} | Device = {device}")
print(f"Chế độ Data-Centric: Crop Foreground = {args.crop_foreground} | Neutral Gray BG = {args.remove_background}")


## Bước 3: Load Data & Data-Centric Foreground Extraction

Để bứt phá giới hạn hội tụ, chúng ta tích hợp cơ chế **Data-Centric AI**:
1. **Trimap Segmentation BBox Crop:** Tự động phát hiện toạ độ bao quanh toàn bộ cơ thể và khuôn mặt của thú cưng, cắt bỏ viền cảnh nền dư thừa bên ngoài.
2. **Background Neutralization:** Khử toàn bộ các pixel cảnh nền (sofa, bãi cỏ, sàn nhà, người ôm) thành màu xám trung tính $. Trong không gian chuẩn hóa ImageNet, giá trị này ánh xạ về $pprox 0$, triệt tiêu hoàn toàn gradient nhiễu của môi trường.
3. **Micro-Texture Augmentation:** Kết hợp CLAHE, Bilateral Filter và Part-level Cropping để tối ưu hóa việc nhận diện giống fine-grained.


In [ ]:
from PIL import Image
import torchvision.transforms.functional as TF

class OxfordPetForegroundDataset(OxfordPetDataset):
    """
    OxfordPetForegroundDataset:
    Tự động khai thác file Trimap Segmentation để:
    1. Xác định Bounding Box chặt bao quanh thú cưng (khuôn mặt, tai, thân, chân)
    2. Cắt bỏ (crop) các vùng cảnh nền dư thừa bên ngoài với padding an toàn (4%)
    3. Xóa các pixel nền (background pixels = 2) thành màu xám trung tính (128, 128, 128)
       nhằm loại bỏ 100% rác môi trường (ghế sofa, người bế, bãi cỏ, thảm),
       giúp mô hình tập trung tối đa vào hình thái và hoa văn đặc trưng của giống.
    """
    def __init__(self, *args, remove_background=True, crop_foreground=True, bg_color=(128, 128, 128), **kwargs):
        super().__init__(*args, **kwargs)
        self.remove_background = remove_background
        self.crop_foreground = crop_foreground
        self.bg_color = bg_color
        self.trimaps_dir = os.path.join(self.root_dir, "annotations", "trimaps")

    def __getitem__(self, idx):
        img_path, target, class_id, spec_id, img_name = self.samples[idx]
        with Image.open(img_path) as img:
            img = img.convert("RGB")

        trimap_path = os.path.join(self.trimaps_dir, f"{img_name}.png")
        if os.path.exists(trimap_path) and (self.remove_background or self.crop_foreground):
            try:
                trimap = Image.open(trimap_path)
                if trimap.size != img.size:
                    trimap = trimap.resize(img.size, Image.NEAREST)

                t_arr = np.array(trimap)
                img_arr = np.array(img)

                # Trimap: 1 = Foreground (Pet), 2 = Background, 3 = Contour/Boundary
                fg_mask = (t_arr == 1) | (t_arr == 3)

                if self.crop_foreground:
                    y_indices, x_indices = np.where(fg_mask)
                    if len(y_indices) > 0 and len(x_indices) > 0:
                        h, w = t_arr.shape
                        pad_y = int(0.04 * h)
                        pad_x = int(0.04 * w)
                        ymin = max(0, y_indices.min() - pad_y)
                        ymax = min(h, y_indices.max() + pad_y)
                        xmin = max(0, x_indices.min() - pad_x)
                        xmax = min(w, x_indices.max() + pad_x)

                        img_arr = img_arr[ymin:ymax, xmin:xmax]
                        t_arr = t_arr[ymin:ymax, xmin:xmax]

                if self.remove_background:
                    bg_mask = (t_arr == 2)
                    img_arr[bg_mask] = self.bg_color

                img = Image.fromarray(img_arr)
            except Exception:
                pass

        if self.transform is not None:
            tensor = self.transform(img)
        else:
            tensor = TF.to_tensor(img)

        return tensor, target, {
            "image_name": img_name,
            "breed_name": self.class_names[class_id],
            "species": "Cat" if spec_id == 0 else "Dog",
            "class_id": class_id,
            "species_id": spec_id
        }

train_transform = get_transforms(img_size=args.img_size, is_train=True, use_part_aug=True, use_clahe=True)
test_transform = get_transforms(img_size=args.img_size, is_train=False)

trainset = OxfordPetForegroundDataset(
    root_dir=args.data_dir, split="trainval", task=args.task, 
    transform=train_transform,
    crop_foreground=args.crop_foreground,
    remove_background=args.remove_background
)
testset = OxfordPetForegroundDataset(
    root_dir=args.data_dir, split="test", task=args.task, 
    transform=test_transform,
    crop_foreground=args.crop_foreground,
    remove_background=args.remove_background
)

classes = trainset.class_names
num_classes = len(classes)

trainloader = DataLoader(trainset, batch_size=args.batch_size, shuffle=True, num_workers=2, pin_memory=(device.type == "cuda"))
testloader = DataLoader(testset, batch_size=args.batch_size, shuffle=False, num_workers=2, pin_memory=(device.type == "cuda"))

print(f">> Khởi tạo thành công Dataset với kỹ thuật Data-Centric Foreground Extraction:")
print(f"- Số mẫu huấn luyện (Trainval): {len(trainset)}")
print(f"- Số mẫu kiểm thử (Test): {len(testset)}")
print(f"- Số lớp phân loại chi tiết: {num_classes} giống thú cưng")
print(f"- Cắt viền vật thể (Crop Foreground): {args.crop_foreground}")
print(f"- Khử cảnh nền thành xám trung tính (Remove BG): {args.remove_background}")


## Bước 4: Kiến trúc Model From Scratch


In [ ]:
# ==============================================================================
# KIẾN TRÚC PETNET-HYBRID V3 (From Scratch - Không sử dụng Pretrained Weights)
# Đề xuất: Dual-Stage Architecture (Conv-SE ở tầng nông + MBConv ở tầng sâu)
# ==============================================================================

class SqueezeExcitation(nn.Module):
    """Channel Attention Squeeze-and-Excitation chuẩn mực của EfficientNet & SENet."""
    def __init__(self, in_channels: int, se_channels: int):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(in_channels, se_channels, bias=False),
            nn.SiLU(inplace=True),
            nn.Linear(se_channels, in_channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, _, _ = x.shape
        w = self.fc(x).view(b, c, 1, 1)
        return x * w


class DropPath(nn.Module):
    """Stochastic Depth per sample cho mạng sâu."""
    def __init__(self, drop_prob: float = 0.0):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if not self.training or self.drop_prob == 0.0:
            return x
        keep_prob = 1.0 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
        random_tensor.floor_()
        return x.div(keep_prob) * random_tensor


class ResidualConvSEBlock(nn.Module):
    """
    Stage 1 & 2 Block: Dense Residual Convolution với SE.
    Gradient cực mạnh, giúp mô hình nắm bắt ngay kết cấu lông & màu sắc trong các epoch đầu.
    """
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1, drop_path: float = 0.0):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.silu = nn.SiLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.se = SqueezeExcitation(out_channels, max(1, out_channels // 4))

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.AvgPool2d(stride, stride) if stride > 1 else nn.Identity(),
                nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()
        self.drop_path = DropPath(drop_path) if drop_path > 0.0 else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        res = self.shortcut(x)
        out = self.silu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        return self.silu(res + self.drop_path(out))


class MBConvBlock(nn.Module):
    """
    Stage 3 & 4 Block: Inverted Residual Bottleneck với Depthwise Conv & SE.
    Không gian đặc trưng mở rộng 4x, triệt tiêu hoàn toàn overfitting ở các tầng sâu.
    """
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1, expand_ratio: int = 4, kernel_size: int = 3, drop_path: float = 0.0):
        super().__init__()
        hidden_dim = in_channels * expand_ratio

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.AvgPool2d(stride, stride) if stride > 1 else nn.Identity(),
                nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()

        padding = (kernel_size - 1) // 2
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, hidden_dim, kernel_size=1, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.SiLU(inplace=True),
            nn.Conv2d(hidden_dim, hidden_dim, kernel_size=kernel_size, stride=stride, padding=padding, groups=hidden_dim, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.SiLU(inplace=True),
            SqueezeExcitation(hidden_dim, max(1, in_channels // 4)),
            nn.Conv2d(hidden_dim, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels)
        )
        self.drop_path = DropPath(drop_path) if drop_path > 0.0 else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.shortcut(x) + self.drop_path(self.conv(x))


class PartAttention(nn.Module):
    """Residual Part-Attention Module đặt tại đầu ra Stage 3 & 4."""
    def __init__(self, channels: int):
        super().__init__()
        self.spatial = nn.Sequential(
            nn.Conv2d(channels, channels // 4, kernel_size=1, bias=False),
            nn.BatchNorm2d(channels // 4),
            nn.SiLU(inplace=True),
            nn.Conv2d(channels // 4, 1, kernel_size=3, padding=1, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x * (1.0 + self.spatial(x))


class PetNetHybridV3(nn.Module):
    """
    Proposed Architecture: PetNet-Hybrid V3 (From Scratch).
    Dung lượng tối ưu: ~4.65M params (Cân bằng hoàn hảo sức mạnh & tốc độ).
    """
    def __init__(self, num_classes: int = 37, in_channels: int = 3, drop_rate: float = 0.25):
        super().__init__()
        # Stem: 224x224 -> 112x112
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.SiLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.SiLU(inplace=True)
        )

        # Stage 1: 112x112 -> 56x56 (64 channels) - Residual Conv + SE
        self.stage1 = nn.Sequential(
            ResidualConvSEBlock(64, 64, stride=2, drop_path=0.03),
            ResidualConvSEBlock(64, 64, stride=1, drop_path=0.03)
        )
        # Stage 2: 56x56 -> 28x28 (128 channels) - Residual Conv + SE
        self.stage2 = nn.Sequential(
            ResidualConvSEBlock(64, 128, stride=2, drop_path=0.06),
            ResidualConvSEBlock(128, 128, stride=1, drop_path=0.06)
        )
        # Stage 3: 28x28 -> 14x14 (224 channels) - Multi-Scale MBConv (3x3 & 5x5)
        self.stage3 = nn.Sequential(
            MBConvBlock(128, 224, stride=2, expand_ratio=4, kernel_size=3, drop_path=0.1),
            MBConvBlock(224, 224, stride=1, expand_ratio=4, kernel_size=5, drop_path=0.1),
            MBConvBlock(224, 224, stride=1, expand_ratio=4, kernel_size=3, drop_path=0.1)
        )
        self.stage3_att = PartAttention(224)

        # Stage 4: 14x14 -> 7x7 (352 channels) - Multi-Scale MBConv (3x3 & 5x5)
        self.stage4 = nn.Sequential(
            MBConvBlock(224, 352, stride=2, expand_ratio=4, kernel_size=3, drop_path=0.15),
            MBConvBlock(352, 352, stride=1, expand_ratio=4, kernel_size=5, drop_path=0.15)
        )
        self.stage4_att = PartAttention(352)

        # Multi-scale Fusion Head (GAP + GMP kết hợp Stage 3 và Stage 4)
        in_feat = (224 * 2) + (352 * 2)  # 448 + 704 = 1152 features
        self.head = nn.Sequential(
            nn.BatchNorm1d(in_feat),
            nn.Dropout(drop_rate),
            nn.Linear(in_feat, 512),
            nn.SiLU(inplace=True),
            nn.Dropout(drop_rate / 2),
            nn.Linear(512, num_classes)
        )

        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        s3 = self.stage3_att(self.stage3(x))
        s4 = self.stage4_att(self.stage4(s3))

        gap3 = F.adaptive_avg_pool2d(s3, 1).flatten(1)
        gmp3 = F.adaptive_max_pool2d(s3, 1).flatten(1)
        gap4 = F.adaptive_avg_pool2d(s4, 1).flatten(1)
        gmp4 = F.adaptive_max_pool2d(s4, 1).flatten(1)

        feat = torch.cat([gap3, gmp3, gap4, gmp4], dim=1)
        return self.head(feat)

model = PetNetHybridV3(num_classes=num_classes).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Khởi tạo thành công PetNet-Hybrid V3 (From Scratch)!")
print(f"- Tổng số tham số (Trainable Parameters): {total_params:,} (~{total_params/1e6:.2f}M)")
print(f"- Cấu trúc Dual-Stage: Residual Conv-SE ở tầng nông + Multi-Scale MBConv ở tầng sâu.")


## Bước 5: Huấn luyện (Training & Checkpoints)


In [ ]:
os.makedirs(args.checkpoint_dir, exist_ok=True)

# Label Smoothing nhẹ (0.05) giúp ranh giới 37 lớp dứt khoát hơn
criterion = nn.CrossEntropyLoss(label_smoothing=args.label_smoothing)
optimizer = optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs, eta_min=1e-6)

best_ckpt = os.path.join(args.checkpoint_dir, "hybrid_petnet_v3_best.pth")
start_epoch = 1
best_acc = 0.0
train_losses, test_losses = [], []
train_accs, test_accs = [], []

# Cơ chế Resume tự động nếu đã có checkpoint tốt nhất (tương thích chuẩn PyTorch 2.6)
if os.path.exists(best_ckpt):
    ckpt = torch.load(best_ckpt, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    start_epoch = ckpt["epoch"] + 1
    best_acc = ckpt["best_acc"]
    history = ckpt["history"]
    train_losses, test_losses = history["train_loss"], history["test_loss"]
    train_accs, test_accs = history["train_acc"], history["test_acc"]
    print(f"Đã phục hồi trạng thái từ Epoch {ckpt['epoch']} với Best Test Acc: {best_acc:.2f}%")

print(f"Bắt đầu huấn luyện PetNet-Hybrid V3 từ Epoch {start_epoch} đến Epoch {args.epochs}...")

for epoch in range(start_epoch, args.epochs + 1):
    # Training Loop
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for inputs, targets, _ in trainloader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(targets).sum().item()
        total += targets.size(0)

    train_loss = running_loss / total
    train_acc = 100.0 * correct / total

    # Validation Loop
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, targets, _ in testloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            running_loss += loss.item() * inputs.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(targets).sum().item()
            total += targets.size(0)

    test_loss = running_loss / total
    test_acc = 100.0 * correct / total
    scheduler.step()

    train_losses.append(train_loss)
    test_losses.append(test_loss)
    train_accs.append(train_acc)
    test_accs.append(test_acc)

    current_lr = optimizer.param_groups[0]["lr"]
    print(f"Epoch {epoch:02d} | LR: {current_lr:.6f} | Train Loss: {train_loss:.3f} | Train Acc: {train_acc:.2f}% | Test Loss: {test_loss:.3f} | Test Acc: {test_acc:.2f}%")

    if test_acc >= best_acc:
        best_acc = test_acc
        ckpt_data = {
            "epoch": epoch,
            "model_name": "hybrid_petnet_v3",
            "total_params": total_params,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_acc": float(best_acc),
            "history": {
                "train_loss": train_losses, "test_loss": test_losses,
                "train_acc": train_accs, "test_acc": test_accs
            }
        }
        torch.save(ckpt_data, best_ckpt)

print(f"Huấn luyện hoàn tất! Best Test Acc đạt được: {best_acc:.2f}%")


## Bước 6: Trực quan hóa & Xuất kết quả (Plots & Metrics)


In [ ]:
os.makedirs(args.plots_dir, exist_ok=True)
epochs_range = range(1, len(train_losses) + 1)

# 1. Đường cong Loss & Accuracy theo Epoch
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, train_losses, "o-", label="Train Loss", color="#2b5c8f", lw=2, markersize=3)
plt.plot(epochs_range, test_losses, "s-", label="Test Loss", color="#e76f51", lw=2, markersize=3)
plt.title("HÌNH 1: ĐƯỜNG CONG MẤT MÁT (LOSS CURVES) - PETNET-HYBRID V3", fontweight="bold", fontsize=11)
plt.xlabel("Epoch", fontweight="bold")
plt.ylabel("Loss", fontweight="bold")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, train_accs, "o-", label="Train Acc", color="#2a9d8f", lw=2, markersize=3)
plt.plot(epochs_range, test_accs, "s-", label="Test Acc", color="#e9c46a", lw=2, markersize=3)
plt.title("HÌNH 2: ĐƯỜNG CONG ĐỘ CHÍNH XÁC (ACCURACY CURVES) - PETNET-HYBRID V3", fontweight="bold", fontsize=11)
plt.xlabel("Epoch", fontweight="bold")
plt.ylabel("Accuracy (%)", fontweight="bold")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()

plt.tight_layout()
curve_path = os.path.join(args.plots_dir, "hybrid_petnet_v3_curves.png")
plt.savefig(curve_path, dpi=300)
plt.show()

# 2. Nạp lại trọng số tốt nhất trước khi đánh giá chuẩn & TTA (Tương thích PyTorch 2.6)
if os.path.exists(best_ckpt):
    best_ckpt_data = torch.load(best_ckpt, map_location=device, weights_only=False)
    model.load_state_dict(best_ckpt_data["model_state_dict"])
    print(f">> Đã nạp trọng số tốt nhất: Epoch {best_ckpt_data['epoch']:02d} (Best Raw Acc: {best_ckpt_data['best_acc']:.2f}%)")

model.eval()

# 3. Đánh giá kiểm thử với Test-Time Augmentation (TTA)
all_preds_raw, all_preds_tta, all_targets = [], [], []

with torch.no_grad():
    for inputs, targets, _ in testloader:
        inputs = inputs.to(device)
        targets = targets.numpy()

        # Dự đoán chuẩn (No TTA)
        logits_orig = model(inputs)
        probs_orig = F.softmax(logits_orig, dim=1)
        preds_raw = probs_orig.max(1)[1].cpu().numpy()

        # Dự đoán TTA (Ensemble Original + Horizontally Flipped View)
        inputs_flip = torch.flip(inputs, dims=[-1])
        logits_flip = model(inputs_flip)
        probs_flip = F.softmax(logits_flip, dim=1)
        probs_tta = (probs_orig + probs_flip) / 2.0
        preds_tta = probs_tta.max(1)[1].cpu().numpy()

        all_preds_raw.extend(preds_raw)
        all_preds_tta.extend(preds_tta)
        all_targets.extend(targets)

acc_raw = 100.0 * np.mean(np.array(all_preds_raw) == np.array(all_targets))
acc_tta = 100.0 * np.mean(np.array(all_preds_tta) == np.array(all_targets))

print("\n================ KẾT QUẢ ĐỐI SÁNH PETNET-HYBRID V3 ================")
print(f"1. Test Accuracy chuẩn (Không dùng TTA) : {acc_raw:.2f}%")
print(f"2. Test Accuracy tối ưu với TTA (Ensemble Views): {acc_tta:.2f}%")
print(f">> Mức tăng trưởng đạt được nhờ TTA     : +{acc_tta - acc_raw:.2f}% tuyệt đối!")
print("===================================================================\n")

# 4. Ma trận nhầm lẫn (Confusion Matrix) trên 37 loài với TTA
cm = confusion_matrix(all_targets, all_preds_tta)
breed_names = [name.replace("_", " ").title() for name in testset.class_names]

plt.figure(figsize=(15, 13))
im = plt.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
plt.title(f"HÌNH 3: MA TRẬN NHẦM LẪN (CONFUSION MATRIX) - PETNET-HYBRID V3 VỚI TTA (ACC: {acc_tta:.2f}%)", fontweight="bold", fontsize=12, pad=15)
cbar = plt.colorbar(im, fraction=0.046, pad=0.04)
cbar.set_label("Số lượng ảnh dự đoán", fontweight="bold", fontsize=11)

tick_marks = np.arange(len(breed_names))
plt.xticks(tick_marks, breed_names, rotation=90, fontsize=8)
plt.yticks(tick_marks, breed_names, fontsize=8)

plt.xlabel("Nhãn Dự Đoán (Predicted Breed - TTA)", fontweight="bold", fontsize=11, labelpad=10)
plt.ylabel("Nhãn Thực Tế (Ground Truth Breed)", fontweight="bold", fontsize=11, labelpad=10)
plt.tight_layout()
cm_path = os.path.join(args.plots_dir, "hybrid_petnet_v3_confusion_matrix.png")
plt.savefig(cm_path, dpi=300, bbox_inches="tight")
plt.show()

# 5. Xuất file tóm tắt chỉ số định lượng
metrics_summary = {
    "model_name": "Proposed PetNet-Hybrid V3 (Dual-Stage Residual-ConvSE & MBConv)",
    "total_params": total_params,
    "batch_size": args.batch_size,
    "epochs_trained": len(train_losses),
    "final_train_loss": round(train_losses[-1], 3) if train_losses else None,
    "final_train_acc": round(train_accs[-1], 2) if train_accs else None,
    "final_test_loss": round(test_losses[-1], 3) if test_losses else None,
    "final_test_acc": round(test_accs[-1], 2) if test_accs else None,
    "best_test_acc_raw": round(acc_raw, 2),
    "best_test_acc_tta": round(acc_tta, 2),
    "delta_tta_gain": round(acc_tta - acc_raw, 2)
}
metrics_file = os.path.join(args.plots_dir, "hybrid_petnet_v3_metrics.json")
with open(metrics_file, "w") as f:
    json.dump(metrics_summary, f, indent=2)
print(f"Đã lưu kết quả đối sánh định lượng tại: {metrics_file}")
